# DharmaShield: Stateful Trust & Safety RL Environment Demo

This notebook guides you through setting up, inspecting, and running the **DharmaShield** reinforcement learning environment. It allows you to run both local heuristic baselines and the full Qwen-72B LLM agent via the HuggingFace Router.

### 1. Installation & Environment Verification
Ensure all package dependencies and the local vendored  package are installed.

In [ ]:
# Install required packages
!pip install -q fastapi uvicorn pydantic httpx openai python-dotenv pyyaml pytest gradio matplotlib
# Install the local openenv package
!pip install -q -e vendor/openenv

### 2. Manual Environment Exploration
Reset the environment, view state parameters, and take manual actions to observe the Safe Harbour delta and reward calculations.

In [ ]:
from dharma_shield.environment import DharmaShieldEnvironment
from dharma_shield.models import DharmaShieldAction

# Initialize the stateful environment
env = DharmaShieldEnvironment()

# Reset to start a task
obs = env.reset("upi-scam-triage")
print("=== Initial Observation ===")
print(f"Task: {env.current_task_id}")
print(f"Current Moderation Item Content: {obs.current_item.text}")
print(f"Active Policy Hints: {obs.active_rule_hints}")
print(f"Safe Harbour Status: {obs.safe_harbour_status}")

In [ ]:
# Simulate taking a compliant action
action = DharmaShieldAction(
    decision="remove",
    rule_cited="IT_2021_3_1_b_iii",
    evidence_signals=obs.current_item.evidence_signals[:2],
    confidence=0.9,
    reason="UPI scam pattern matched NPCI financial fraud templates.",
    notify_user=True,
    target_account=obs.current_item.target_account
)

next_obs, reward_info, done, info = env.step(action)

print("=== Step Result ===")
print(f"Step Reward: {reward_info.step_reward:.3f}")
print(f"Reward Details: {reward_info.model_dump()}")
print(f"Safe Harbour Level: {next_obs.safe_harbour_status}")
print(f"Done: {done}")

### 3. Run the Local Heuristic Baseline (Free / No Token Required)
This executes the local baseline loop using rules and heuristics.

In [ ]:
# Run local baseline
!python inference.py

### 4. Run the Qwen-72B-Instruct Agent via HuggingFace Router
Provide your HuggingFace API key to run the full LLM evaluation loop.

In [ ]:
import os
import getpass
from dotenv import load_dotenv

# Load token from .env if it exists
load_dotenv()

hf_token = os.getenv("HF_TOKEN")
if not hf_token:
    hf_token = getpass.getpass("Enter your HuggingFace API Token: ")

os.environ["HF_TOKEN"] = hf_token
os.environ["REQUIRE_HF_ROUTER"] = "true"
os.environ["VERBOSE"] = "true"

# Run the LLM inference evaluation
!python inference.py

### 5. Performance Comparison Visualization
Plot a comparative graph showing the Heuristic Baseline vs. State-of-the-Art LLM (Qwen-72B) across the tasks.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

tasks = ["UPI Scam", "SGI Media", "CIB Networks", "Child Safety"]
heuristics = [0.491, 0.146, 0.206, 0.591]
qwen_72b = [0.858, 0.962, 0.569, 0.880]
target_score = [1.0, 1.0, 1.0, 1.0]

x = np.arange(len(tasks))
width = 0.25

fig, ax = plt.subplots(figsize=(10, 6))
rects1 = ax.bar(x - width, heuristics, width, label="Local Heuristics (Baseline)", color="#e74c3c")
rects2 = ax.bar(x, qwen_72b, width, label="Qwen 2.5 72B (SOTA LLM)", color="#3498db")
rects3 = ax.bar(x + width, target_score, width, label="Target Optimal Score", color="#2ecc71", alpha=0.5, linestyle="--", edgecolor="black")

ax.set_ylabel("Evaluation Score")
ax.set_title("Performance Comparison across DharmaShield Tasks")
ax.set_xticks(x)
ax.set_xticklabels(tasks)
ax.legend()

fig.tight_layout()
plt.show()